In [1]:
from dotenv import load_dotenv
import os 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_tavily import TavilySearch
from langchain.tools import tool
from pydantic import BaseModel , Field 
from langchain_classic.agents.react.agent import create_react_agent
from langchain_classic.agents import AgentExecutor

from langchain_classic import hub #Think of it like GitHub, but specifically for LangChain prompts and chains.
#so this is a place where you can find structured prompts and chains 

load_dotenv(override=True)

api_key = os.getenv("LLM_API_KEY")
if not api_key:
    print("LLM_API_KEY not found")
else:
    print("Api key exists")

Api key exists


In [2]:
from typing import List
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda

In [3]:
class Source(BaseModel):
    """Schema for a source used by the agent"""

    url : str = Field(description="The url of the source")

class AgentResponse(BaseModel):
    """Scehama for agent response with answer and sources"""

    answer : str = Field(description="Agent's answer to query")
    sources : List[Source] = Field(
        default_factory=list , description="List of sources used by the agent to answer the query"
    )


In [4]:
react_prompt_with_format_instructions = """
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}

IMPORTANT:
You must respond with exactly ONE of the following:
- An Action and Action Input
- OR a Final Answer
Never both in the same response.

"""



In [5]:
output_parser = PydanticOutputParser(pydantic_object=AgentResponse)

In [6]:
react_prompt_with_format_instructions = PromptTemplate(
    template = react_prompt_with_format_instructions,
    input_variables = ["input" , "agent_scratchpad" , "tools" , "tool_names" , "format_instructions"]
).partial(format_instructions = output_parser.get_format_instructions())
#“The output must be a valid JSON object with the following fields…”
#format_instructions = rules that tell the LLM how its FINAL output must be structured so the output parser can parse it

In [7]:
tools = [TavilySearch()]
llm = ChatOpenAI(model="openai/gpt-5", temperature=0 , api_key=os.getenv("LLM_API_KEY") , base_url=os.getenv("BASE_URL"))
agent = create_react_agent(
    llm, 
    tools,  
    react_prompt_with_format_instructions
)

agent_executor = AgentExecutor(agent = agent , tools = tools , verbose = True )

In [68]:
chain = agent_executor

result = chain.invoke(
    input={
        "input":"search for 3 Ai jobs in india"
    }
)



> Entering new AgentExecutor chain...
Action: tavily_search
Action Input: AI jobs in India Machine Learning Engineer Data Scientist NLP jobs India 2026{'query': 'AI jobs in India Machine Learning Engineer Data Scientist NLP jobs India 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://in.linkedin.com/jobs/data-science-natural-language-processing-nlp-jobs', 'title': '475 Data Science Natural Language Processing Nlp jobs in India', 'content': '402 Data Science Natural Language Processing Nlp Jobs in India (13 new) · AI/ML Developer - Artificial Intelligence · AI / ML Engineer · Senior Machine Learning', 'score': 0.99965405, 'raw_content': None}, {'url': 'https://www.foundit.in/search/ml-gen-ai-nlp-llm-jobs-in-india', 'title': '331 Ml Gen Ai Nlp Llm Job vacancies in India 2026 - Foundit', 'content': 'Ml Gen Ai Nlp Llm Jobs in India · Data Science · ML with Gen AI · Data Scientist Agentic AI / Graph/ LLM Science , Senior Associate · Data Scient

In [69]:
print(result["input"])
print(result["output"])

search for 3 Ai jobs in india
- Cognizant — Gen AI Engineer — Bangalore, Karnataka, India
  Link: https://careers.cognizant.com/india-en/jobs/00065735601/gen-ai-engineer/

- Google Cloud — Field Solutions Architect, Generative AI — India
  Link (job board listing): https://www.foundit.in/search/generative-ai-jobs

- KeyValue Software Systems — Machine Learning Engineer — Cochin/Kochi, India
  Link (job board listing): https://www.hirist.tech/c/ai-ml-jobs
